In [ ]:
#!pip install biopython transformers torch_geometric

In [2]:
import os
import requests
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.PDB.PDBExceptions import PDBConstructionWarning
from scipy.spatial import Voronoi, distance_matrix
from transformers import AutoTokenizer, EsmModel
from torch_geometric.data import Data

In [3]:
import os
path_main = '/Data'
os.chdir(path_main)

In [ ]:

warnings.simplefilter('ignore', PDBConstructionWarning)

try:
    import gudhi
    GUDHI_AVAILABLE = True
except ImportError:
    GUDHI_AVAILABLE = False
    #print("The Gudhi library is not installed. The empty topology will be filled with zeros.")


# Pipeline settings

CSV_PATH = "Test_A_B.xlsx"               # Source table
RAW_PDB_DIR = "./raw_pdbs"              # Where to download raw PDBs
PROCESSED_DIR = "./benchmark_graphs"    # Where to save finished .pt graphs
ESM_MODEL_NAME = "facebook/esm2_t33_650M_UR50D" # ESM-2 model

os.makedirs(RAW_PDB_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading ESM-2({ESM_MODEL_NAME}) на {device}...")
tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)
esm_model = EsmModel.from_pretrained(ESM_MODEL_NAME).to(device)
esm_model.eval()

# General Dictionary of Amino Acid Translations
D3TO1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

In [ ]:
# PDB download function
def download_pdb(pdb_id):
    """
    Downloads the PDB file if it is not on the disk or if it is damaged (0 bytes)
    """
    file_path = os.path.join(RAW_PDB_DIR, f"{pdb_id}.pdb")

    # Checking if a file exists and its size is greater than 0 bytes
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        return file_path
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with open(file_path, 'wb') as f:
                f.write(response.content)
            return file_path
        else:
            return None
    except Exception:
        return None

# Extraction of ESM-2 embeddings
@torch.no_grad()
def get_sequence_embedding(sequence):
    """Running the sequence through ESM-2"""
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = esm_model(**inputs)
    last_hidden_state = outputs.last_hidden_state

    residue_embeddings = last_hidden_state[0, 1:-1, :]
    return residue_embeddings.cpu()




In [ ]:
# Voronoi Geometry Calculation Class
class PDBGeometryProcessor:
    def __init__(self, contact_cutoff_intra=8.0, contact_cutoff_inter=10.0):
        self.parser = PDBParser(QUIET=True)
        self.cutoff_intra = contact_cutoff_intra
        self.cutoff_inter = contact_cutoff_inter

    def _extract_coords_and_seq(self, pdb_path, receptor_chains, ligand_chains):
        structure = self.parser.get_structure("complex", pdb_path)
        rec_coords, lig_coords = [], []
        seq_rec, seq_lig = "", ""

        for chain in structure[0]:
            chain_id = chain.get_id()
            if chain_id not in receptor_chains and chain_id not in ligand_chains:
                continue

            for residue in chain:
                if residue.get_id()[0] != ' ' or 'CA' not in residue:
                    continue
                resname = residue.get_resname()
                if resname in D3TO1:
                    coord = residue['CA'].get_coord()
                    if chain_id in receptor_chains:
                        rec_coords.append(coord)
                        seq_rec += D3TO1[resname]
                    elif chain_id in ligand_chains:
                        lig_coords.append(coord)
                        seq_lig += D3TO1[resname]

        return np.array(rec_coords), np.array(lig_coords), seq_rec, seq_lig

    def _polygon_area_3d(self, vertices):
        if len(vertices) < 3: return 0.0
        v0 = vertices[0]
        area = 0.0
        for i in range(1, len(vertices) - 1):
            area += np.linalg.norm(np.cross(vertices[i] - v0, vertices[i+1] - v0))
        return 0.5 * area

    def _compute_voronoi_areas(self, coords):
        if len(coords) < 4: return {}
        vor = Voronoi(coords)
        contact_areas = {}
        for point_indices, vertex_indices in zip(vor.ridge_points, vor.ridge_vertices):
            if -1 in vertex_indices: continue
            p1, p2 = point_indices
            area = self._polygon_area_3d(vor.vertices[vertex_indices])
            contact_areas[(p1, p2)] = area
            contact_areas[(p2, p1)] = area
        return contact_areas

    def _compute_persistent_homology(self, coords):
        if not GUDHI_AVAILABLE or len(coords) < 4: return [0.0, 0.0, 0.0]
        try:
            alpha_complex = gudhi.AlphaComplex(points=coords)
            simplex_tree = alpha_complex.create_simplex_tree()
            persistence = simplex_tree.persistence()
            h1_intervals = [p[1] for p in persistence if p[0] == 1 and p[1][1] != float('inf')]
            if len(h1_intervals) > 0:
                h1_lifespans = [top - birth for birth, top in h1_intervals]
                return [np.mean(h1_lifespans), np.max(h1_lifespans), float(len(h1_intervals))]
            return [0.0, 0.0, 0.0]
        except Exception: return [0.0, 0.0, 0.0]

    def process_structure(self, pdb_path, receptor_chains, ligand_chains):
        coords_rec, coords_lig, seq_rec, seq_lig = self._extract_coords_and_seq(pdb_path, receptor_chains, ligand_chains)

        if len(coords_rec) == 0 or len(coords_lig) == 0:
            raise ValueError("No C-alpha atoms found")

        N_rec, N_lig = len(coords_rec), len(coords_lig)
        total_res = N_rec + N_lig
        coords_complex = np.vstack([coords_rec, coords_lig])

        dist_matrix = distance_matrix(coords_complex, coords_complex)
        voronoi_areas = self._compute_voronoi_areas(coords_complex)

        # Intra-chain
        adj_intra = (dist_matrix < self.cutoff_intra) & (~np.eye(total_res, dtype=bool))
        adj_intra[:N_rec, N_rec:] = False
        adj_intra[N_rec:, :N_rec] = False
        edge_index_intra = torch.tensor(np.argwhere(adj_intra).T, dtype=torch.long)

        # Inter-chain
        adj_inter = np.zeros((total_res, total_res), dtype=bool)
        adj_inter_np = dist_matrix[:N_rec, N_rec:] < self.cutoff_inter
        adj_inter[:N_rec, N_rec:] = adj_inter_np
        adj_inter[N_rec:, :N_rec] = adj_inter_np.T
        edge_index_inter = torch.tensor(np.argwhere(adj_inter).T, dtype=torch.long)

        edge_index = torch.cat([edge_index_intra, edge_index_inter], dim=1)
        E_total = edge_index.shape[1]

        edge_attr = torch.zeros((E_total, 4), dtype=torch.float32)
        rows, cols = edge_index[0], edge_index[1]

        edge_attr[:, 0] = torch.tensor(dist_matrix[rows, cols], dtype=torch.float32)
        edge_attr[:, 1] = 1.0 / (edge_attr[:, 0] + 1e-6)

        for i in range(E_total):
            edge_attr[i, 2] = voronoi_areas.get((int(rows[i]), int(cols[i])), 0.0)

        is_inter_edge = (rows < N_rec) & (cols >= N_rec) | (rows >= N_rec) & (cols < N_rec)
        edge_attr[is_inter_edge, 3] = 1.0

        inter_mask = np.any(adj_inter_np, axis=1)
        lig_mask = np.any(adj_inter_np, axis=0)
        interface_coords = np.vstack([coords_rec[inter_mask], coords_lig[lig_mask]])
        global_features = torch.tensor([self._compute_persistent_homology(interface_coords)], dtype=torch.float32)

        return {
            'edge_index': edge_index,
            'edge_attr': edge_attr,
            'global_features': global_features,
            'seq_rec': seq_rec,
            'seq_lig': seq_lig
        }


In [ ]:
# Main loop
if __name__ == "__main__":
    df = pd.read_excel(CSV_PATH).dropna(subset=['Complex_ID'])
    processor = PDBGeometryProcessor()

    success_count, error_count = 0, 0

    @torch.no_grad()
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Generating Benchmark Graphs"):
        pdb_id = str(row['Complex_ID']).strip().lower()
        save_path = os.path.join(PROCESSED_DIR, f"{pdb_id}.pt")

        if os.path.exists(save_path):
            success_count += 1
            continue

        try:
            pdb_path = download_pdb(pdb_id)
            if not pdb_path: raise ValueError(f"Failed to download {pdb_id}")

            rec_chains = row['Receptor Chains']
            lig_chains = row['Ligand Chains']
            if not rec_chains: raise ValueError("Receptor and ligand could not be identified")

            # Voronoi Mathematics and Sequence Extraction
            geom = processor.process_structure(pdb_path, rec_chains, lig_chains)

            # ESM-2 embeddings
            emb_rec = get_sequence_embedding(geom['seq_rec'])
            flag_rec = torch.tensor([[1.0, 0.0]], dtype=torch.float).repeat(emb_rec.shape[0], 1)
            x_rec = torch.cat([emb_rec, flag_rec], dim=1) # [N_rec, 1282]

            emb_lig = get_sequence_embedding(geom['seq_lig'])
            flag_lig = torch.tensor([[0.0, 1.0]], dtype=torch.float).repeat(emb_lig.shape[0], 1)
            x_lig = torch.cat([emb_lig, flag_lig], dim=1) # [N_lig, 1282]

            # Gluing of knots
            x_final = torch.cat([x_rec, x_lig], dim=0)

            # Final assembly of the graph
            data = Data(
                x=x_final,
                edge_index=geom['edge_index'],
                edge_attr=geom['edge_attr'],
                global_features=geom['global_features'],
                item_id=pdb_id
            )

            # Saving graphs
            torch.save(data, save_path)
            success_count += 1

        except Exception as e:
            print(f"\nError {pdb_id}: {e}")
            error_count += 1


    print("Dataset generation completed")
    print("-"*50)
    print(f"Successfully:{success_count}")
    print(f"Errors: {error_count}")